# EEG · Pooled Balanced Random Forest

Trains a Random Forest on a **randomly sampled, class-balanced** pool built from all
training trials. Unlike the LOTO approach, the K-sample train set is drawn from the full
multi-trial pool. The CV split during Optuna uses **GroupKFold by trial** to prevent
same-trial autocorrelated windows from leaking across folds.

```
Trials 0–7  →  build all windows  →  pool  →  random sample K balanced  →  RF
Trials 8–9  →  all windows (unbalanced)                                  →  evaluate
```

| Step | Details |
|---|---|
| Pool | All causal windows from trials 0–7 |
| Sampling | K/2 from class 0, K/2 from class 1 (random, no replacement) |
| Optuna CV | GroupKFold(3) by trial ID — no same-trial leakage |
| Threshold | Youden's J on train CV holdout |
| Test | Trials 8 & 9, full unbalanced windows, never seen during training |

## 1 · Imports & Setup

In [1]:
import sys

sys.path.insert(0, "..")
from src.preprocessing import build_windows, load_trial, make_horizon_labels

In [2]:
import warnings

warnings.filterwarnings("ignore")
import json
import os

import joblib
import numpy as np
import optuna
import pandas as pd
from optuna.pruners import MedianPruner
from optuna.samplers import TPESampler
from sklearn.calibration import CalibratedClassifierCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    classification_report,
    confusion_matrix,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import StratifiedKFold

optuna.logging.set_verbosity(optuna.logging.WARNING)

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt

from src.preprocessing import build_windows, load_trial, make_horizon_labels

DARK = "#0a0e17"
CARD = "#111827"
EDGE = "#1f2937"
TEAL = "#00e5cc"
CORAL = "#ff4f5e"
GOLD = "#ffc947"
LIME = "#a8ff3e"
WHITE = "#f0f4ff"
GRAY = "#6b7280"

print("All imports OK.")

All imports OK.


## 2 · Configuration

In [3]:
FS     = 250               # original recording rate
DECIM  = 4                 # downsampling factor (1 = off)
STRIDE = 1                 # window stride — keep 1 here; sample_balanced handles autocorrelation
FS_EFF = FS // DECIM       # effective rate after decimation
HORIZON = int(1.0 * FS_EFF)  # anticipatory label lookahead: 1000 ms (full Bereitschaftspotential)

SUBJECT = 9
TACHE = "spt"
DATA_PATH = f"../data/subject{SUBJECT}/"

TRAIN_TRIALS = list(range(8))  # 0–7 → source pool
TEST_TRIALS = [8, 9]  # never seen during training

K = 100_000  # total balanced training samples (K/2 per class)
T_MAX_S = 2.0                    # max look-back in seconds
T_MAX   = int(T_MAX_S * FS_EFF)  # in samples at FS_EFF
N_TRIALS_OPTUNA = 100
INNER_FOLDS = 3
RANDOM_SEED = 42

print(f"FS / FS_EFF  : {FS} Hz → {FS_EFF} Hz  (DECIM={DECIM})")
print(f"HORIZON      : {HORIZON} samples  ({HORIZON/FS_EFF*1000:.0f} ms lookahead)")
print(f"Train trials : {TRAIN_TRIALS}")
print(f"Test  trials : {TEST_TRIALS}")
print(f"K            : {K:,}  ({K//2:,} per class)")
print(f"T_MAX        : {T_MAX} samples  ({T_MAX/FS_EFF*1000:.0f} ms)")

FS / FS_EFF  : 250 Hz → 62 Hz  (DECIM=4)
HORIZON      : 62 samples  (1000 ms lookahead)
Train trials : [0, 1, 2, 3, 4, 5, 6, 7]
Test  trials : [8, 9]
K            : 100,000  (50,000 per class)
T_MAX        : 124 samples  (2000 ms)


## 3 · Load All Train Trials

Each trial is band-pass filtered (0.1–60 Hz) independently by `load_trial`.
All trials are concatenated with a `trial_id` vector so windows cannot
straddle two trial boundaries during pre-computation.

In [4]:
print("Loading train trials …")
EEG_COLS = None
X_parts, y_parts, g_parts = [], [], []

for t in TRAIN_TRIALS:
    df, cols = load_trial(
        SUBJECT,
        TACHE,
        t,
        DATA_PATH,
        eeg_cols=EEG_COLS,
        apply_filter=True,
        decimate=DECIM,
        lp=0.5,
        hp=8
    )
    if EEG_COLS is None:
        EEG_COLS = cols
    X_parts.append(df[EEG_COLS].values.astype(np.float32))
    y_t = make_horizon_labels(df["button"].values.astype(int), HORIZON)
    y_parts.append(y_t)
    g_parts.append(np.full(len(df), t, dtype=np.int64))
    bal = df["button"].value_counts().to_dict()
    print(f"  trial {t}: {len(df):>6,} samples  label={bal}")

X_pool = np.concatenate(X_parts, axis=0)
y_pool = np.concatenate(y_parts, axis=0)
g_pool = np.concatenate(g_parts, axis=0)

N_CHANNELS = len(EEG_COLS)
n_pos = (y_pool == 1).sum()
n_neg = (y_pool == 0).sum()
n_each_max = min(K // 2, n_pos)

print(f"\nCombined pool : {X_pool.shape}")
print(f"Class balance (horizon labels) : {{0: {n_neg:,}, 1: {n_pos:,}}}")
print(f"Minority class: {n_pos:,}  →  max K/2 per class = {n_each_max:,}")
print(f"Effective K   : {2 * n_each_max:,}  ({n_each_max:,} per class)")

Loading train trials …
Setting up band-pass filter from 0.5 - 8 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 0.50
- Lower transition bandwidth: 0.50 Hz (-6 dB cutoff frequency: 0.25 Hz)
- Upper passband edge: 8.00 Hz
- Upper transition bandwidth: 2.00 Hz (-6 dB cutoff frequency: 9.00 Hz)
- Filter length: 1651 samples (6.604 s)

  trial 0:  5,085 samples  label={0: 4679, 1: 406}
Setting up band-pass filter from 0.5 - 8 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 0.50
- Lower transition bandwidth: 0.50 Hz (-6 dB cutoff frequency: 0.25 Hz)
- Upper passband edge: 8.00 Hz
- Upp

## 4 · Pre-compute Windows at T_MAX

Build **all** causal windows at the maximum look-back (`T_MAX`) once.
Optuna then slices the last `T` timesteps per trial rather than rebuilding
windows from scratch — orders of magnitude faster.

`build_windows` with `groups=g_pool` skips any window that would straddle
two different trials, preventing boundary leakage.

In [5]:
print(f"Pre-computing windows at T_MAX={T_MAX} …")
_Xw, _yw, _gw = build_windows(
    X_pool, y_pool, T_MAX, groups=g_pool, per_window_norm=False, stride=STRIDE
)
# build_windows with groups → (M, C, T_MAX) channels-first; transpose for time-axis slicing
_X_max = _Xw.transpose(0, 2, 1)  # (M, T_MAX, C)
print(f"  Pool windows : {_X_max.shape}  memory={_X_max.nbytes/1e6:.1f} MB")
print(f"  Class balance: {{0: {(_yw==0).sum():,}, 1: {(_yw==1).sum():,}}}")


def get_windows_for_T(T: int):
    """Slice the last T timesteps + vectorised per-window normalisation."""
    X = _X_max[:, T_MAX - T :, :].copy()  # (M, T, C)
    mu = X.mean(axis=1, keepdims=True)
    sd = X.std(axis=1, keepdims=True) + 1e-8
    return ((X - mu) / sd).reshape(len(_yw), T * N_CHANNELS), _yw, _gw


def sample_balanced(
    X: np.ndarray, y: np.ndarray, g: np.ndarray, K: int, rng: np.random.Generator
):
    """Randomly draw min(K/2, n_minority) samples from each class, preserving group labels."""
    idx0, idx1 = np.where(y == 0)[0], np.where(y == 1)[0]
    n_each = min(K // 2, len(idx0), len(idx1))
    sel = np.concatenate(
        [
            rng.choice(idx0, n_each, replace=False),
            rng.choice(idx1, n_each, replace=False),
        ]
    )
    rng.shuffle(sel)
    return X[sel], y[sel], g[sel]


def feature_names(eeg_cols, T):
    return [f"{ch}_lag{t}" for ch in eeg_cols for t in range(T)]


# Sanity check
X_chk, y_chk, g_chk = get_windows_for_T(32)
rng_chk = np.random.default_rng(0)
X_b, y_b, g_b = sample_balanced(X_chk, y_chk, g_chk, 1000, rng_chk)
print(f"\nSanity check T=32 → all: {X_chk.shape}  balanced 1000: {X_b.shape}")
print(f"  y_b balance: {{0: {(y_b==0).sum()}, 1: {(y_b==1).sum()}}}")
print(f"  g_b unique trials: {np.unique(g_b)}")

Pre-computing windows at T_MAX=124 …
  Pool windows : (33304, 124, 16)  memory=264.3 MB
  Class balance: {0: 15,527, 1: 17,777}

Sanity check T=32 → all: (33304, 512)  balanced 1000: (1000, 512)
  y_b balance: {0: 500, 1: 500}
  g_b unique trials: [0 1 2 3 4 5 6 7]


## 5 · Optuna Hyperparameter Search

Each Optuna trial:
1. Slices windows for the proposed `T` (fast — no rebuild)
2. Samples a fresh K-balanced set (different seed per trial for robustness)
3. Evaluates with `GroupKFold(3)` split by trial ID — prevents temporally
   autocorrelated windows from the same trial leaking across folds

| Hyperparameter | Range |
|---|---|
| **T** | 4 – T_MAX (log) |
| `n_estimators` | 50 – 500 (log) |
| `max_depth` | None, 5, 10, 20, 30 |
| `min_samples_split` | 2 – 20 |
| `min_samples_leaf` | 1 – 10 |
| `max_features` | sqrt, log2, 0.1, 0.3 |
| `class_weight` | balanced, None |

In [ ]:
from sklearn.model_selection import GroupKFold

cv_inner = GroupKFold(n_splits=INNER_FOLDS)


def objective(trial):
    T = trial.suggest_int("T", max(4, FS_EFF // 16), T_MAX, log=True)

    n_estimators = trial.suggest_int("n_estimators", 10, 1000, log=True)
    max_depth = trial.suggest_categorical("max_depth", [None, 5, 10, 20, 30])
    min_samples_split = trial.suggest_int("min_samples_split", 2, 20)
    min_samples_leaf = trial.suggest_int("min_samples_leaf", 1, 10)
    max_features = trial.suggest_categorical("max_features", ["sqrt", "log2", 0.1, 0.3])
    class_weight = trial.suggest_categorical("class_weight", ["balanced", None])

    # Fast slice + fresh balanced sample for this trial
    X_all, y_all, g_all = get_windows_for_T(T)
    rng = np.random.default_rng(trial.number + RANDOM_SEED)
    X_bal, y_bal, g_bal = sample_balanced(X_all, y_all, g_all, K, rng)

    clf = RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        min_samples_split=min_samples_split,
        min_samples_leaf=min_samples_leaf,
        max_features=max_features,
        class_weight=class_weight,
        n_jobs=-1,
        random_state=42,
    )

    fold_aucs = []
    for fold, (tr_idx, va_idx) in enumerate(cv_inner.split(X_bal, y_bal, groups=g_bal)):
        clf.fit(X_bal[tr_idx], y_bal[tr_idx])
        probs = clf.predict_proba(X_bal[va_idx])[:, 1]
        auc = roc_auc_score(y_bal[va_idx], probs)
        fold_aucs.append(auc)
        trial.report(np.mean(fold_aucs), step=fold)
        if trial.should_prune():
            raise optuna.TrialPruned()

    return float(np.mean(fold_aucs))


study = optuna.create_study(
    direction="maximize",
    sampler=TPESampler(seed=RANDOM_SEED),
    pruner=MedianPruner(n_startup_trials=10, n_warmup_steps=0),
)
study.optimize(objective, n_trials=N_TRIALS_OPTUNA, show_progress_bar=True)

print(f"\nBest trial : #{study.best_trial.number}")
print(f"Best AUC   : {study.best_value:.4f}")
print("\nBest params:")
for k, v in study.best_params.items():
    print(f"  {k:22s}: {v}")


  0%|                                                                                                                                                           | 0/100 [00:00<?, ?it/s]

## 6 · Extract Best Config & Build Final Training Set

Resample once with `RANDOM_SEED` to get the reproducible final training set.

In [ ]:
p = study.best_params

T = p["T"]
n_estimators = p["n_estimators"]
max_depth = p["max_depth"]
min_samples_split = p["min_samples_split"]
min_samples_leaf = p["min_samples_leaf"]
max_features = p["max_features"]
class_weight = p["class_weight"]

X_all, y_all, g_all = get_windows_for_T(T)
X_bal, y_bal, _ = sample_balanced(
    X_all, y_all, g_all, K, np.random.default_rng(RANDOM_SEED)
)
feat_cols = feature_names(EEG_COLS, T)

print(f"Best T         : {T} samples  ({T/FS_EFF*1000:.0f} ms look-back)")
print(f"Training set   : {X_bal.shape}  →  {y_bal.shape}")
print(
    f"  class balance: {{0: {(y_bal==0).sum()}, 1: {(y_bal==1).sum()}}}  (exactly balanced)"
)
print(f"n_estimators   : {n_estimators}")
print(f"max_depth      : {max_depth}")
print(f"min_samples_split / leaf : {min_samples_split} / {min_samples_leaf}")
print(f"max_features   : {max_features}")
print(f"class_weight   : {class_weight}")


## 7 · Final Evaluation on Held-Out Trials

Train on the balanced K-sample set, evaluate on the full (unbalanced) test pool.
The test set was **never seen** during Optuna tuning or threshold selection.

In [ ]:
print("Loading test trials …")
X_test_parts, y_test_parts, g_test_parts = [], [], []

for t in TEST_TRIALS:
    df_t, _ = load_trial(
        SUBJECT,
        TACHE,
        t,
        DATA_PATH,
        eeg_cols=EEG_COLS,
        apply_filter=True,
        decimate=DECIM,
        lp=0.5,
        hp=8,
    )
    X_test_parts.append(df_t[EEG_COLS].values.astype(np.float32))
    y_t = make_horizon_labels(df_t["button"].values.astype(int), HORIZON)
    y_test_parts.append(y_t)
    g_test_parts.append(np.full(len(df_t), t, dtype=np.int64))
    bal = df_t["button"].value_counts().to_dict()
    print(f"  trial {t}: {len(df_t):>6,} samples  label={bal}")

X_test_pool = np.concatenate(X_test_parts, axis=0)
y_test_pool = np.concatenate(y_test_parts, axis=0)
g_test_pool = np.concatenate(g_test_parts, axis=0)

print(f"\nTest pool : {X_test_pool.shape}")
print(f"Class balance (horizon labels) : {{0: {(y_test_pool==0).sum():,}, 1: {(y_test_pool==1).sum():,}}}")

In [ ]:
# ── Train on balanced set ─────────────────────────────────────────────────
best_clf = RandomForestClassifier(
    n_estimators=n_estimators,
    max_depth=max_depth,
    min_samples_split=min_samples_split,
    min_samples_leaf=min_samples_leaf,
    max_features=max_features,
    class_weight=class_weight,
    n_jobs=-1,
    random_state=RANDOM_SEED,
)
best_clf.fit(X_bal, y_bal)
importances = best_clf.feature_importances_
print("RF fitted on balanced training set.")

# ── CV AUC on training pool (self-check) ──────────────────────────────────
cv_check = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)
cv_aucs = []
for tr_idx, va_idx in cv_check.split(X_bal, y_bal):
    clf_tmp = RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        min_samples_split=min_samples_split,
        min_samples_leaf=min_samples_leaf,
        max_features=max_features,
        class_weight=class_weight,
        n_jobs=-1,
        random_state=RANDOM_SEED,
    )
    clf_tmp.fit(X_bal[tr_idx], y_bal[tr_idx])
    probs_va = clf_tmp.predict_proba(X_bal[va_idx])[:, 1]
    cv_aucs.append(roc_auc_score(y_bal[va_idx], probs_va))
print(
    f"CV AUC on balanced train set (5-fold): {np.mean(cv_aucs):.4f} ± {np.std(cv_aucs):.4f}"
)

# ── Youden's J threshold on CV holdout probs ──────────────────────────────
cv_probs = np.zeros(len(y_bal), dtype=np.float32)
for tr_idx, va_idx in cv_check.split(X_bal, y_bal):
    clf_tmp = RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        min_samples_split=min_samples_split,
        min_samples_leaf=min_samples_leaf,
        max_features=max_features,
        class_weight=class_weight,
        n_jobs=-1,
        random_state=RANDOM_SEED,
    )
    clf_tmp.fit(X_bal[tr_idx], y_bal[tr_idx])
    cv_probs[va_idx] = clf_tmp.predict_proba(X_bal[va_idx])[:, 1]

fpr_cv, tpr_cv, thr_cv = roc_curve(y_bal, cv_probs)
best_thresh = float(thr_cv[np.argmax(tpr_cv - fpr_cv)])
print(f"Optimal threshold (Youden's J) : {best_thresh:.3f}")

# ── Build test windows (full, unbalanced) ─────────────────────────────────
_Xw_test, y_test, g_test = build_windows(
    X_test_pool, y_test_pool, T, groups=g_test_pool
)
X_test = _Xw_test.transpose(0, 2, 1).reshape(-1, N_CHANNELS * T)

probs_test = best_clf.predict_proba(X_test)[:, 1]
preds_test = (probs_test >= best_thresh).astype(int)
auc_test = roc_auc_score(y_test, probs_test)
acc_test = (preds_test == y_test).mean()

print("\n" + "=" * 60)
print("  Held-out Test Results (trials 8 & 9)")
print("=" * 60)
print(f"  Windows  : {X_test.shape[0]:,}")
print(f"  AUC      : {auc_test:.4f}")
print(f"  Accuracy : {acc_test:.4f}  (threshold={best_thresh:.3f})")
print()
print(
    classification_report(y_test, preds_test, target_names=["not-pressed", "pressed"])
)

# Per-trial breakdown
for t in TEST_TRIALS:
    mask_t = g_test == t
    auc_t = roc_auc_score(y_test[mask_t], probs_test[mask_t])
    acc_t = (preds_test[mask_t] == y_test[mask_t]).mean()
    print(f"  trial {t}: AUC={auc_t:.4f}  Acc={acc_t:.4f}  n={mask_t.sum():,}")


## 8 · Plots

In [ ]:
%matplotlib inline
# ── A: Optuna history — T vs AUC + optimisation curve ─────────────────────
trials_df = study.trials_dataframe(attrs=("number", "value", "params", "state"))
comp = trials_df[trials_df["state"] == "COMPLETE"].sort_values("number")
vals = comp["value"].values
nums = comp["number"].values
best_sf = np.maximum.accumulate(vals)

fig, axes = plt.subplots(1, 2, figsize=(14, 4), facecolor=DARK)

# left: T vs AUC
ax = axes[0]
ax.set_facecolor(CARD)
for sp in ax.spines.values():
    sp.set_color(EDGE)
col_T = "params_T"
if col_T in comp.columns:
    T_vals = comp[col_T].dropna().values
    auc_vals = comp.loc[comp[col_T].notna(), "value"].values
    sc = ax.scatter(
        T_vals / FS_EFF * 1000,
        auc_vals,
        c=auc_vals,
        cmap="plasma",
        s=55,
        alpha=0.85,
        zorder=3,
    )
    ax.axvline(
        T / FS_EFF * 1000,
        color=CORAL,
        lw=2.5,
        ls="--",
        label=f"Best T={T} ({T/FS_EFF*1000:.0f} ms)",
    )
    plt.colorbar(sc, ax=ax, label="AUC")
ax.set_xlabel("Window T  (ms)", color=WHITE, fontsize=10)
ax.set_ylabel("Validation AUC", color=WHITE, fontsize=10)
ax.set_title("Look-back Duration vs AUC", color=WHITE, fontsize=11, fontweight="bold")
ax.tick_params(colors=WHITE)
ax.yaxis.grid(True, color=EDGE)
ax.legend(facecolor=CARD, labelcolor=WHITE, edgecolor=EDGE)

# right: optimisation history
ax = axes[1]
ax.set_facecolor(CARD)
for sp in ax.spines.values():
    sp.set_color(EDGE)
ax.scatter(nums, vals, color=TEAL, alpha=0.45, s=25, zorder=3, label="Trial AUC")
ax.plot(nums, best_sf, color=GOLD, lw=2.5, zorder=4, label="Best so far")
ax.axhline(
    best_sf[-1], color=CORAL, lw=1, ls="--", label=f"Best={best_sf[-1]:.4f}", alpha=0.8
)
ax.set_xlabel("Trial", color=WHITE, fontsize=10)
ax.set_ylabel("Validation AUC", color=WHITE, fontsize=10)
ax.set_title("Optimisation History", color=WHITE, fontsize=11, fontweight="bold")
ax.tick_params(colors=WHITE)
ax.yaxis.grid(True, color=EDGE)
ax.legend(facecolor=CARD, labelcolor=WHITE, edgecolor=EDGE)

plt.suptitle(
    f"Optuna Search — Pooled Balanced RF  (K={K:,})",
    color=WHITE,
    fontsize=13,
    fontweight="bold",
)
plt.tight_layout()
plt.show()


In [ ]:
# ── B: Feature importance heatmap (channel × lag) + per-channel bar ───────
imp_matrix = importances.reshape(N_CHANNELS, T)
ch_importance = imp_matrix.sum(axis=1)

fig, axes = plt.subplots(
    1, 2, figsize=(16, 5), facecolor=DARK, gridspec_kw={"width_ratios": [3, 1]}
)

ax = axes[0]
ax.set_facecolor(CARD)
for sp in ax.spines.values():
    sp.set_color(EDGE)
im = ax.imshow(imp_matrix, aspect="auto", cmap="viridis", interpolation="nearest")
plt.colorbar(im, ax=ax, label="Gini Importance")
ax.set_yticks(range(N_CHANNELS))
ax.set_yticklabels(EEG_COLS, color=WHITE, fontsize=9)
ax.set_xlabel("Lag (samples before target,  0 = most recent)", color=WHITE, fontsize=10)
ax.set_title(
    "Feature Importance — Channel × Lag", color=WHITE, fontsize=12, fontweight="bold"
)
ax.tick_params(colors=WHITE)
best_ch, best_lag = np.unravel_index(imp_matrix.argmax(), imp_matrix.shape)
ax.add_patch(
    plt.Rectangle(
        (best_lag - 0.5, best_ch - 0.5), 1, 1, fill=False, edgecolor=CORAL, lw=2.5
    )
)
ax.text(
    best_lag,
    best_ch - 0.6,
    f"peak: {EEG_COLS[best_ch]} lag={best_lag}",
    color=CORAL,
    fontsize=8,
    ha="center",
)

ax = axes[1]
ax.set_facecolor(CARD)
for sp in ax.spines.values():
    sp.set_color(EDGE)
order = np.argsort(ch_importance)[::-1]
colours = [GOLD if i == order[0] else TEAL for i in range(N_CHANNELS)]
ax.barh(
    range(N_CHANNELS),
    ch_importance[order[::-1]],
    color=colours[::-1],
    edgecolor=EDGE,
    alpha=0.9,
)
ax.set_yticks(range(N_CHANNELS))
ax.set_yticklabels([EEG_COLS[i] for i in order[::-1]], color=WHITE, fontsize=9)
ax.set_xlabel("Total importance (sum over lags)", color=WHITE, fontsize=9)
ax.set_title("Per-Channel", color=WHITE, fontsize=12, fontweight="bold")
ax.tick_params(colors=WHITE)
ax.xaxis.grid(True, color=EDGE)

plt.suptitle(
    f"Feature Importance (T={T}, trained on balanced K={len(y_bal):,})",
    color=WHITE,
    fontsize=13,
    fontweight="bold",
)
plt.tight_layout()
plt.show()

In [ ]:
# ── C: Test probability timeline ──────────────────────────────────────────
fig, axes = plt.subplots(3, 1, figsize=(18, 7), sharex=True, facecolor=DARK)
t_ax = np.arange(len(probs_test))
split = int((g_test == TEST_TRIALS[0]).sum())

ax = axes[0]
ax.set_facecolor(CARD)
ax.plot(t_ax, probs_test, color=TEAL, lw=0.6, label="P(pressed)")
ax.axhline(
    best_thresh, color=CORAL, lw=1.2, ls="--", label=f"Threshold={best_thresh:.3f}"
)
ax.axvline(split, color=GOLD, lw=1.5, ls=":", label=f"Trial {TEST_TRIALS[1]} start")
ax.set_ylim(0, 1)
ax.set_ylabel("P(pressed)", color=WHITE, fontsize=9)
ax.tick_params(colors=WHITE)
ax.legend(facecolor=CARD, labelcolor=WHITE, edgecolor=EDGE, fontsize=8)
for sp in ax.spines.values():
    sp.set_color(EDGE)

ax = axes[1]
ax.set_facecolor(CARD)
ax.plot(
    t_ax, preds_test, drawstyle="steps-post", color=CORAL, lw=1.5, label="Predicted"
)
ax.plot(
    t_ax, y_test, drawstyle="steps-post", color=WHITE, lw=0.8, alpha=0.5, label="True"
)
ax.axvline(split, color=GOLD, lw=1.5, ls=":")
ax.set_ylim(-0.1, 1.1)
ax.set_yticks([0, 1])
ax.set_ylabel("Label", color=WHITE, fontsize=9)
ax.tick_params(colors=WHITE)
ax.legend(facecolor=CARD, labelcolor=WHITE, edgecolor=EDGE, fontsize=8)
for sp in ax.spines.values():
    sp.set_color(EDGE)

ax = axes[2]
ax.set_facecolor(CARD)
errors = (preds_test != y_test).astype(float)
ax.fill_between(t_ax, errors, color=GOLD, alpha=0.7, step="post", label="Error")
ax.axvline(split, color=GOLD, lw=1.5, ls=":")
ax.set_ylim(0, 1.2)
ax.set_yticks([])
ax.set_ylabel("Error", color=WHITE, fontsize=9)
ax.set_xlabel("Window index (trials 8 & 9 concatenated)", color=WHITE, fontsize=9)
ax.tick_params(colors=WHITE)
ax.legend(facecolor=CARD, labelcolor=WHITE, edgecolor=EDGE, fontsize=8)
for sp in ax.spines.values():
    sp.set_color(EDGE)

fig.suptitle(
    f"Held-Out Test (trials {TEST_TRIALS}) · AUC={auc_test:.4f}  Acc={acc_test:.4f}",
    color=WHITE,
    fontsize=13,
    fontweight="bold",
)
plt.tight_layout()
plt.show()

In [ ]:
# ── D: Confusion matrix ───────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(5, 4), facecolor=DARK)
ax.set_facecolor(CARD)
for sp in ax.spines.values():
    sp.set_color(EDGE)

cm = confusion_matrix(y_test, preds_test)
ConfusionMatrixDisplay(cm, display_labels=["not-pressed", "pressed"]).plot(
    ax=ax, colorbar=False, cmap="YlOrRd"
)
ax.set_title(
    f"Confusion Matrix — Test (trials {TEST_TRIALS})",
    color=WHITE,
    fontsize=10,
    fontweight="bold",
)
ax.tick_params(colors=WHITE)
ax.xaxis.label.set_color(WHITE)
ax.yaxis.label.set_color(WHITE)
for txt in ax.texts:
    txt.set_color("black")
    txt.set_fontsize(13)
plt.tight_layout()
plt.show()

## 9 · Results Summary & Save

In [ ]:
top3 = np.argsort(ch_importance)[::-1][:3]
print("=" * 65)
print("  EEG Pooled Balanced RF — Results")
print("=" * 65)
print(f"  Train trials  : {TRAIN_TRIALS}")
print(f"  Test  trials  : {TEST_TRIALS}")
print(f"  Look-back T   : {T} samples  ({T/FS_EFF*1000:.0f} ms)")
print(
    f"  Training set  : K={len(y_bal):,}  ({(y_bal==1).sum():,} pressed / {(y_bal==0).sum():,} not-pressed)"
)
print(f"  Features      : {N_CHANNELS} ch × {T} lags = {N_CHANNELS*T}")
print()
print(f"  Optuna best AUC (inner CV) : {study.best_value:.4f}")
print(f"  CV AUC on balanced set     : {np.mean(cv_aucs):.4f} ± {np.std(cv_aucs):.4f}")
print(f"  Optimal threshold          : {best_thresh:.3f}")
print()
print(f"  TEST AUC       : {auc_test:.4f}")
print(f"  TEST Accuracy  : {acc_test:.4f}")
print()
print("  Best RF config:")
print(f"    n_estimators      : {n_estimators}")
print(f"    max_depth         : {max_depth}")
print(f"    min_samples_split : {min_samples_split}")
print(f"    min_samples_leaf  : {min_samples_leaf}")
print(f"    max_features      : {max_features}")
print(f"    class_weight      : {class_weight}")
print()
print("  Top-3 channels (by total importance):")
for i in top3:
    print(f"    {EEG_COLS[i]:6s}  {ch_importance[i]:.4f}")
print("=" * 65)


In [ ]:
SAVE_PATH = f"../data/subject{SUBJECT}/model_pooled"
os.makedirs(SAVE_PATH, exist_ok=True)

joblib.dump(best_clf, f"{SAVE_PATH}/eeg_rf_model.joblib")

params_to_save = {
    "T": int(T),
    "n_estimators": int(n_estimators),
    "max_depth": max_depth,
    "min_samples_split": int(min_samples_split),
    "min_samples_leaf": int(min_samples_leaf),
    "max_features": max_features,
    "class_weight": class_weight,
    "K": int(len(y_bal)),
    "optuna_auc": float(study.best_value),
    "cv_auc_mean": float(np.mean(cv_aucs)),
    "cv_auc_std": float(np.std(cv_aucs)),
    "test_auc": float(auc_test),
    "test_acc": float(acc_test),
    "best_thresh": float(best_thresh),
    "n_channels": int(N_CHANNELS),
    "channel_names": EEG_COLS,
    "fs": int(FS_EFF),
    "train_trials": TRAIN_TRIALS,
    "test_trials": TEST_TRIALS,
    "per_window_norm": True,
    "random_seed": int(RANDOM_SEED),
}
with open(f"{SAVE_PATH}/eeg_rf_params.json", "w") as f:
    json.dump(params_to_save, f, indent=2)

print("Saved:")
print(f"  {SAVE_PATH}/eeg_rf_model.joblib")
print(f"  {SAVE_PATH}/eeg_rf_params.json")
print(f"  thresh={best_thresh:.3f}  test_auc={auc_test:.4f}  K={len(y_bal):,}")
